# Predicting Snow-Water Equivalent at Watershed Scale using Machine Learning on SNOTEL and PRISM datasets

## Introduction

Last winter was one of the strangest I can remember in my career as a ski instructor in Montana. Over the entire Christmas break - typically a cold, snowy period - it rained for almost my entire commute. Instead of the ground disappearing under a blanket of white, as it usually does for months, the grass in my yard was visible almost the entire winter. Among my peers, the conversation invariably included some version of "This is so strange." and "What bad will the summer be?" Indeed, this is a pressing question for people across the Rocky Mountain region of the US. Many places just had one of the warmest winters ever recorded, with extremely low snowpack. Now, as summer hits, water resources are dwindling, and forests across the Rockies are drying out and catching flame.

As climate changes progresses, there are expected changes to snowpack dynamics in the Rockies. The 2025-2026 winter was perhaps a peak into the future; one where precipitation falls more as rain than snow, snow melts earlier, and the water flowing into streams and reservoirs comes at different times or with different amounts. These potential changes are paramount to understand for stakeholders who depend on snow - municipalities whose drinking water comes from snowpack, ski resorts, wildland firefighters, or infrastructure engineers are just a few examples. In reality, almost anyone living in the mountainous Rockies might be impacted by changes to snowpack. Assuming that water is mainly stored as snowpack, gaining clarity on snowpack dynamics across a region is crucial to understanding the available water resources. Given that snowpack-derived water users likely come from a variety of backgrounds, understanding of snowpack must be fairly accessible and easy to update. Models or forecasts that can be run on a personal computer are advantageous relative to models that require high-performance computing.

This study is informed by the context surrounding snowpack dynamics in the Rockies. In order to bridge the gap between point SNOTEL records and regional-scale models, either hind-cast or future-looking, this study proposes the use of machine learning (ML) to predict snowpack data based on easily available temperature and precipitation data. This project develops such a ML model that can predict snow-water equivalent (SWE) across a spatial area based on precipitation and temperature data. Once trained and validated on historical data, the ML model could be applied to future climate projections.

## Methods

#### 2.1: Machine Learning and Snowpack
This study is hardly the first to utilize machine learning in efforts to model snowpack; many other researchers have tackled this issue. Machine learning has shown promise in capturing snowpack dynamics. Researchers have used a variety of methods, such as deep learning, neural networks, decision trees, gradient boosting, and hybrid models ([Alabi et al. 2026](https://doi.org/10.1175/AIES-D-25-0021.1), [Duan et al. 2024](https://doi.org/10.1029/2023WR035009), [O'Flaherty 2025](https://scholarworks.montana.edu/server/api/core/bitstreams/9947cdce-776a-484e-a108-b5b82093cd04/content), [Ouyang et al. 2026](https://doi.org/10.3390/w18101243), [Steele et al. 2024](https://doi.org/10.1029/2023WR035805)). 

One challenge of utilizing ML to model SWE is algorithm selection. Hybrid models, such as that presented by Steele et al. 2024, show promise, as they combine the algorithmic power of ML with physics-bound calculation. Deep learning or neural network algorithms also show strong performance. However, hybrid model and deep learning/neural networks require significant computing power to run, limiting the applicability outside academic or large-scale corporate use. Decision-tree models, on the other hand, are easily deployed on personal computer-scale equipment, increasing their potential use, and show similar or better performance than computationally expensive approaches. In addition, the strength of decision-tree algorithms can be supplemented by using an ensemble approach. Ensemble approaches seem to be particularly robust, as the strengths and weaknesses inherent in each algorithm can be combined to better capture the full dynamics of snowpack (Alabi et al. 2026, O'Flaherty 2025).

#### 2.2: Study Region
<embed type="text/html" src=C:\Users\raini\Documents\Graduate_School\EDA_Certificate\Summer\snow-drought-modeling\outputs\study_area_sntl_plot.html height="600" width="600">

The study area used for this project is the Missouri Headwaters HUC6 watershed in Southwestern Montana. This watershed is the source of the Missouri River (the second longest river in the US), and contains multiple mountain ranges, like the Gallatin, Madison, Bridger, and Tobacco Root ranges. It contains 28 SNOTEL stations; 25 of which met the criteria for inclusion in the ML model.

#### 2.3: Training Datasets
| Dataset | Access |
| --- | --- |
| SNOTEL historical daily SWE, temp, precip time series | USDA AWDB API |
| BCQC SNOTEL dataset | PNNL website |
| SRTM Digital Elevation Model | TNM Access API |
| PRISM gridded historical climate data | PRISM Climate Group API |

Data used in the ML model are indicated in the table above. SNOTEL data provided the target variable (SWE). Instead of using the raw SNOTEL data from the USGS, a bias-corrected, quality controlled dataset that has managed known issues with the SNOTEL dataset was used ([Sun et al. 2019](https://doi.org/10.1029/2018JD030140), [Yan et al. 2019](https://doi.org/10.1002/2017WR021290)) PRISM data provided climate data used as predictor variables. The PRISM dataset is an interpolated dataset that excels at capturing weather patterns in complex mountainous terrain (Daly et al. 2008), and is freely available at 4km resolution; given the aim of this study, PRISM was the ideal option to provide climate data. Climate data included daily minimum temperature, maximum temperature, and precipitation; rolling windows of weather patterns as well as temporal variables were engineered from the PRISM data.

#### 2.4: Feature Engineering
Research has found that engineering additional features into the climate dataset improves ML algorithm performance. For instance, Alabi et al. 2026 found that day of water year, as well as rolling 7-day windows of mean temperature and total precipitation, helped improve the algorithm accuracy significantly. This is because temporally dependent. A given day's SWE is dictated by everything that's happened in a winter up to that day; just giving a ML model a day's weather data isn't enough for the model to predict with any accuracy. 

Guided by these findings, this study engineered the following features:
- Day of Water Year: how many days after October 1 a given day is. This offers the model some temporal grounding and is a proxy for different accumulation/melt processes throughout the year.
- Cumulative Positive Degree Days: This adds all of the temperature readings that are above freezing, giving the model some sense of how much heat has been added to the snowpack throughout the year.
- Cumulative Precipitation: This tells the model how much precipitation has already fallen at any given day.
- Rolling 7-day Windows of Mean Minimum/Maximum Temperature: Gives the model some memory of antecedent temperature; this can help the model determine if the snowpack is stable or melting.
- Rolling 7-day Window of Total Precipitation: Gives the model some memory of antecedent precipitation; this helps the model know if the snowpack is growing or stable.

#### 2.5: Machine Learning Approach
Informed by the literature on ML and snowpack, this study uses Random Forests (a decision-tree algorithm) to predict SWE. A decision-tree algorithm was chosen for two main reasons: ease of deployment, and demonstrated skill in predicting SWE based on climatic variables. Decision-tree algorithms seem to have similar levels of skill in predicting SWE as more computationally expensive algorithms, but are deployable on a variety of computing equipment. Given the focus on developing a model that could be used by a variety of stakeholders, decision-tree algorithms are the obvious choice. Random Forests was chosen as a simple starting point. There are more advanced decision-tree algorithms (such as gradient boosting algorithms like XGBoost or CatBoost), but Random Forests has been shown to be a dependable algorithms (Alabi et al. 2026), while remaining easy to use. When used in Python, Random Forests accepts common data formats like Pandas DataFrames, and requires just a few lines of code from the scikit-learn package to set up and fit the model; validation functions are likewise user-friendly.

## 3: Results

Training of the machine learning model has shown promising results

## 4: Discussion

## 5: Conclusion and Future Work